In [6]:
# parsing metadata
import gzip
from pathlib import Path

scr =  Path("../data/raw/GSE122198_family.soft.gz")
metadata = {}
with gzip.open(scr, "rt") as f:
    current_sample = None
    for line in f:
        if line.startswith("^SAMPLE"):
            current_sample = line.strip().split(" = ")[1]
            metadata[current_sample] = {}
        elif line.startswith("!Sample_") and current_sample:
            key, val = line.strip().split(" = ", 1)
            metadata[current_sample][key] = val

In [10]:
import scanpy as sc
import pandas as pd

# If it's a dense matrix (genes × cells)
df = pd.read_csv(scr, sep="\t", index_col=0)
adata = sc.AnnData(X=df.T)  # Scanpy expects cells × genes

# Attach metadata
adata.obs_names   # cell barcodes/IDs
adata.var_names_make_unique()   # gene names

/home/ace/miniconda3/envs/jak2/lib/python3.14/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [12]:
# QC
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)

IndexError: Positions outside range of features.